# Cat pcap: giu 300.000 goi DAU TIEN moi file

Duyet **de quy** moi `.pcap` / `.pcapng` trong `/kaggle/input`, cat con `MAX_PACKETS`
goi dau tien, ghi ra `/kaggle/working` voi hau to `_capped_300k`.

## Vi sao giu goi DAU TIEN, khong phai giua hay ngau nhien

1. **Chuoi lien tuc theo thoi gian** — bat buoc cho K-voting / N-consecutive o Viec 5.
   Lay mau rai rac (vd moi 10 goi lay 1) se pha huy cau truc thoi gian ma toan bo
   phan decision strategy dua vao.
2. **Tai lap duoc** — chay lai luon ra dung ket qua, khong co tham so ngau nhien.
3. **Khong them tham so phai bien minh** — cat giua doi hoi chon "diem bat dau",
   la mot lua chon tuy y truoc hoi dong.

**Rui ro da biet**: voi tan cong CHAM (SlowLoris), phan dau file co the la giai doan
thiet lap chu chua phai tan cong chinh. Cell cuoi in **do phu thoi gian** (so giay
ma 300k goi trai dai) va canh bao file nao qua ngan, de ban phat hien truong hop nay.

## Vi sao cat duoc ma khong hong benchmark

Viec 5/6 khong can nhieu du lieu, chi can **du cua so moi lop** va **chuoi lien tuc**.
Thuc te tu lan chay truoc: `Uploading_Attack` chi 1.251 cua so van tinh duoc moi metric,
trong khi `DDoS-UDP_Flood` 793.007 cua so la thua thai. Doi so cua so thua lay them
**file moi** (nhat la Benign — truc chinh de do false-alarm) la doi tot.

Tien le: Realguard (Nguyen et al., Sensors 2022) chon 200.000 goi moi loai tan cong
va 400.000 goi normal tu CIC-IDS2017 de xu ly mat can bang.

## Cau hinh

| Bien | Mac dinh | Y nghia |
|---|---|---|
| `MAX_PACKETS` | `300_000` | so goi dau tien giu lai |
| `NO_CAP_FILES` | `['benign.pcap', ...]` | file **khong** cat, giu nguyen 100% |
| `SUFFIX` | `'_capped_300k'` | hau to ten file dau ra |

File **it hon** `MAX_PACKETS` goi duoc copy nguyen ven (van them hau to cho dong bo ten).

## Cong cu

Uu tien `tcpdump -c N` (co san tren Kaggle, C thuan, nhanh). Neu khong co, tu dong
lui ve `dpkt`. **Khong bao gio cat byte tho** — pcap co global header 24 byte o dau,
cat byte se lam hong file.


In [1]:
import os, sys, glob, time, shutil, subprocess
from datetime import datetime

def _pip(pkg):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)
try:
    import dpkt
except ImportError:
    _pip('dpkt'); import dpkt

# ══════════════════════════ CAU HINH ══════════════════════════
INPUT_DIR  = '/kaggle/input'      # quet DE QUY, tim moi *.pcap / *.pcapng
OUT_DIR    = '/kaggle/working'    # noi ghi file da cat
os.makedirs(OUT_DIR, exist_ok=True)

MAX_PACKETS = 300_000             # giu N goi DAU TIEN moi file
SUFFIX      = '_capped_300k'      # hau to them vao ten file dau ra

# File KHONG cat (giu nguyen 100%). So khop theo ten file, khong phan biet hoa thuong.
# Benign la truc chinh de do false-alarm rate -> giu day du, khong cat.
NO_CAP_FILES = ['benign.pcap', 'benigntraffic.pcap', 'Benign.pcap']

# File nho hon MAX_PACKETS: chi COPY nguyen ven (van them hau to cho dong bo ten).
COPY_IF_SMALLER = True
# ══════════════════════════════════════════════════════════════


def has_cmd(c):
    return shutil.which(c) is not None

HAS_TCPDUMP = has_cmd('tcpdump')
HAS_EDITCAP = has_cmd('editcap')
HAS_CAPINFOS = has_cmd('capinfos')

print('Cong cu san co:')
print(f'  tcpdump : {"co" if HAS_TCPDUMP else "KHONG (se dung dpkt)"}')
print(f'  editcap : {"co" if HAS_EDITCAP else "khong"}')
print(f'  capinfos: {"co" if HAS_CAPINFOS else "khong"}')
print(f'  dpkt    : co (v{dpkt.__version__})')
print()
print(f'INPUT_DIR   : {INPUT_DIR}')
print(f'OUT_DIR     : {OUT_DIR}')
print(f'MAX_PACKETS : {MAX_PACKETS:,} goi DAU TIEN moi file')
print(f'KHONG cat   : {NO_CAP_FILES}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.0/195.0 kB 4.5 MB/s eta 0:00:00
Cong cu san co:
  tcpdump : KHONG (se dung dpkt)
  editcap : khong
  capinfos: khong
  dpkt    : co (v1.9.8)

INPUT_DIR   : /kaggle/input
OUT_DIR     : /kaggle/working
MAX_PACKETS : 300,000 goi DAU TIEN moi file
KHONG cat   : ['benign.pcap', 'benigntraffic.pcap', 'Benign.pcap']


In [2]:
# ═══ HAM DOC THONG TIN + CAT ═══
# Cat bang tcpdump -c (C thuan, nhanh) neu co; neu khong, fallback sang dpkt.
# TUYET DOI khong cat byte tho: file pcap co global header 24 byte o dau,
# cat byte se lam hong file va dpkt/tcpdump se khong doc duoc.

def open_reader(fh):
    """Mo pcap hoac pcapng. Tra ve reader cua dpkt."""
    try:
        return dpkt.pcap.Reader(fh)
    except ValueError:
        fh.seek(0)
        return dpkt.pcapng.Reader(fh)


def pcap_info(path, max_scan=None):
    """Dem goi + lay timestamp dau/cuoi. max_scan=None -> quet het file.

    Luu y: phai doc TUAN TU het file moi biet tong so goi (pcap khong co
    index). Voi file lon day la khau ton thoi gian nhat cua ca notebook.
    """
    n = 0
    ts_first = ts_last = None
    with open(path, 'rb') as f:
        try:
            rd = open_reader(f)
        except Exception as e:
            return {'error': str(e)}
        for ts, _ in rd:
            if ts_first is None:
                ts_first = ts
            ts_last = ts
            n += 1
            if max_scan and n >= max_scan:
                break
    return {'n_packets': n, 'ts_first': ts_first, 'ts_last': ts_last,
            'duration': (ts_last - ts_first) if (ts_first is not None) else 0.0}


def cut_tcpdump(src, dst, n):
    """Cat bang tcpdump -c N. Nhanh nhat, giu nguyen dinh dang."""
    r = subprocess.run(['tcpdump', '-r', src, '-w', dst, '-c', str(n)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'tcpdump loi: {r.stderr.strip()[:200]}')


def cut_dpkt(src, dst, n):
    """Cat bang dpkt (fallback khi khong co tcpdump).

    Ghi ra dinh dang pcap chuan. Giu nguyen linktype cua file goc de khong
    lam sai cach dpkt/extractor bocode tang lien ket o buoc sau.
    """
    with open(src, 'rb') as fi:
        rd = open_reader(fi)
        linktype = getattr(rd, 'datalink', lambda: 1)()
        if callable(linktype):
            linktype = linktype()
        with open(dst, 'wb') as fo:
            wr = dpkt.pcap.Writer(fo, linktype=linktype)
            c = 0
            for ts, buf in rd:
                wr.writepkt(buf, ts=ts)
                c += 1
                if c >= n:
                    break
    return c


def cut_pcap(src, dst, n):
    """Cat n goi dau tien tu src -> dst. Uu tien tcpdump, lui ve dpkt."""
    if HAS_TCPDUMP:
        try:
            cut_tcpdump(src, dst, n)
            return 'tcpdump'
        except Exception as e:
            print(f'      [!] tcpdump that bai ({e}) -> chuyen sang dpkt')
    cut_dpkt(src, dst, n)
    return 'dpkt'


def out_name(src_path):
    """Ten file dau ra: <ten goc><SUFFIX>.pcap (giu duoi goc)."""
    base = os.path.basename(src_path)
    stem, ext = os.path.splitext(base)
    return f'{stem}{SUFFIX}{ext}'


def fmt_ts(ts):
    if ts is None:
        return '-'
    return datetime.fromtimestamp(ts).strftime('%H:%M:%S')


print('Da dinh nghia: pcap_info / cut_pcap / out_name')

Da dinh nghia: pcap_info / cut_pcap / out_name


In [3]:
# ═══ LIET KE FILE + XEM TRUOC (chua cat gi ca) ═══
pcaps = sorted(glob.glob(os.path.join(INPUT_DIR, '**', '*.pcap'), recursive=True) +
               glob.glob(os.path.join(INPUT_DIR, '**', '*.pcapng'), recursive=True))
# bo qua chinh cac file da cat (neu chay lai lan 2)
pcaps = [p for p in pcaps if SUFFIX not in os.path.basename(p)]
assert pcaps, f'Khong tim thay pcap nao trong {INPUT_DIR}'

print(f'Tim thay {len(pcaps)} file pcap.\n')
print(f'{"file":<40}{"kich thuoc":>12}   se lam gi')
print('-' * 78)
plan = []
for p in pcaps:
    base = os.path.basename(p)
    mb = os.path.getsize(p) / 1e6
    if base.lower() in [x.lower() for x in NO_CAP_FILES]:
        act = 'GIU NGUYEN (trong NO_CAP_FILES)'
        mode = 'copy'
    else:
        act = f'cat con {MAX_PACKETS:,} goi dau (neu du)'
        mode = 'cut'
    plan.append({'src': p, 'mode': mode})
    print(f'{base[:38]:<40}{mb:>9.1f} MB   {act}')

print()
print('LUU Y ve thoi gian chay:')
print('  Pcap khong co index -> phai doc TUAN TU het file moi biet tong so goi.')
print('  Voi file vai GB, khau dem nay ton thoi gian hon ca khau cat.')
print('  => Cell sau CHI dem den MAX_PACKETS+1 roi dung (du de biet co can cat khong),')
print('     va dem day du CHI voi file thuc su nho hon nguong.')

Tim thay 14 file pcap.

file                                      kich thuoc   se lam gi
------------------------------------------------------------------------------
BrowserHijacking.pcap                        35.0 MB   cat con 300,000 goi dau (neu du)
DDoS-HTTP_Flood-.pcap                       610.9 MB   cat con 300,000 goi dau (neu du)
DDoS-ICMP_Flood23.pcap                     2048.0 MB   cat con 300,000 goi dau (neu du)
DDoS-ICMP_Fragmentation16.pcap             2048.0 MB   cat con 300,000 goi dau (neu du)
DDoS-PSHACK_Flood12.pcap                   2048.0 MB   cat con 300,000 goi dau (neu du)
DDoS-RSTFINFlood14.pcap                    2048.0 MB   cat con 300,000 goi dau (neu du)
DDoS-SynonymousIP_Flood10.pcap             2048.0 MB   cat con 300,000 goi dau (neu du)
DDoS-UDP_Fragmentation10.pcap              2048.0 MB   cat con 300,000 goi dau (neu du)
DoS-HTTP_Flood.pcap                        2048.0 MB   cat con 300,000 goi dau (neu du)
Mirai-greip_flood15 (1).pcap            

---
## Chay chinh

Toi uu quan trong: **khong dem het** file lon. Pcap khong co index nen phai doc tuan tu
moi biet tong so goi — voi file vai GB, khau dem con ton thoi gian hon khau cat.
Chuong trinh chi doc den `MAX_PACKETS + 1` goi la du biet file co vuot nguong hay khong.


In [4]:
# ═══ XU LY: CAT / COPY ═══
# Toi uu quan trong: KHONG dem het file lon. Chi doc den MAX_PACKETS+1 goi
# la du biet file co vuot nguong hay khong. Neu vuot -> cat luon. Neu khong
# -> luc do so goi da dem chinh la tong so goi that.
#
# Vi sao giu 300k goi DAU TIEN (khong phai giua / ngau nhien):
#   1. Chuoi LIEN TUC theo thoi gian — bat buoc cho K-voting / N-consecutive
#      o Viec 5. Lay mau rai rac se pha huy cau truc thoi gian.
#   2. Tai lap duoc: chay lai luon ra dung ket qua, khong co tham so ngau nhien.
#   3. Khong them tham so "diem bat dau" phai bien minh truoc hoi dong.
# Rui ro da biet: voi tan cong CHAM (vd SlowLoris), phan dau file co the la
# giai doan thiet lap. Bang duoi in DO PHU THOI GIAN (giay giu lai / giay goc)
# de phat hien ngay truong hop do.

rows = []
t_all = time.time()

for i, item in enumerate(plan, 1):
    src, mode = item['src'], item['mode']
    base = os.path.basename(src)
    dst = os.path.join(OUT_DIR, out_name(src))
    t0 = time.time()
    print(f'[{i}/{len(plan)}] {base}')

    # --- dem den nguong (khong dem het file lon) ---
    probe = pcap_info(src, max_scan=MAX_PACKETS + 1)
    if 'error' in probe:
        print(f'      [X] khong doc duoc: {probe["error"][:80]} -> BO QUA\n')
        rows.append({'file': base, 'status': 'ERROR', 'n_in': 0, 'n_out': 0})
        continue
    n_seen = probe['n_packets']
    over = n_seen > MAX_PACKETS

    if mode == 'copy' or not over:
        # giu nguyen: copy file. Voi nhanh nay n_seen CHINH LA tong so goi that
        # (vi vong dem da chay het file ma chua cham nguong).
        reason = 'trong NO_CAP_FILES' if mode == 'copy' else f'chi co {n_seen:,} goi (< nguong)'
        if mode == 'copy':
            # can biet tong so goi that -> phai quet het (Benign co the rat lon)
            full = pcap_info(src)
            n_in = full['n_packets']; dur_in = full['duration']
        else:
            n_in = n_seen; dur_in = probe['duration']
        if COPY_IF_SMALLER or mode == 'copy':
            shutil.copy2(src, dst)
            n_out, dur_out, tool = n_in, dur_in, 'copy'
            status = 'GIU NGUYEN'
        else:
            n_out, dur_out, tool, status = 0, 0, '-', 'BO QUA'
        print(f'      {status} ({reason}) — {n_in:,} goi, {dur_in:.1f}s')
    else:
        # vuot nguong -> cat
        tool = cut_pcap(src, dst, MAX_PACKETS)
        chk = pcap_info(dst)
        n_out, dur_out = chk['n_packets'], chk['duration']
        n_in = f'>{MAX_PACKETS:,}'     # khong dem het -> khong biet chinh xac
        dur_in = None
        status = 'DA CAT'
        print(f'      DA CAT bang {tool}: giu {n_out:,} goi, phu {dur_out:.1f}s '
              f'({fmt_ts(chk["ts_first"])} -> {fmt_ts(chk["ts_last"])})')

    rows.append({'file': base, 'status': status, 'n_in': n_in, 'n_out': n_out,
                 'duration_out_s': round(dur_out, 2),
                 'out_MB': round(os.path.getsize(dst)/1e6, 2) if os.path.exists(dst) else 0,
                 'tool': tool, 'sec': round(time.time()-t0, 1),
                 'out_file': os.path.basename(dst)})
    print(f'      -> {os.path.basename(dst)}  '
          f'({os.path.getsize(dst)/1e6:.1f} MB, {time.time()-t0:.1f}s)\n')

print(f'Tong thoi gian: {time.time()-t_all:.1f}s')

[1/14] BrowserHijacking.pcap
      GIU NGUYEN (chi co 59,821 goi (< nguong)) — 59,821 goi, 1560.0s
      -> BrowserHijacking_capped_300k.pcap  (35.0 MB, 0.7s)

[2/14] DDoS-HTTP_Flood-.pcap
      DA CAT bang dpkt: giu 300,000 goi, phu 292.3s (13:47:00 -> 13:51:52)
      -> DDoS-HTTP_Flood-_capped_300k.pcap  (117.5 MB, 3.6s)

[3/14] DDoS-ICMP_Flood23.pcap
      DA CAT bang dpkt: giu 300,000 goi, phu 12.0s (17:53:56 -> 17:54:08)
      -> DDoS-ICMP_Flood23_capped_300k.pcap  (22.9 MB, 2.7s)

[4/14] DDoS-ICMP_Fragmentation16.pcap
      DA CAT bang dpkt: giu 300,000 goi, phu 580244.6s (20:16:43 -> 13:27:28)
      -> DDoS-ICMP_Fragmentation16_capped_300k.pcap  (265.8 MB, 4.0s)

[5/14] DDoS-PSHACK_Flood12.pcap
      DA CAT bang dpkt: giu 300,000 goi, phu 13.7s (17:46:09 -> 17:46:23)
      -> DDoS-PSHACK_Flood12_capped_300k.pcap  (22.8 MB, 2.6s)

[6/14] DDoS-RSTFINFlood14.pcap
      DA CAT bang dpkt: giu 300,000 goi, phu 14.7s (19:11:25 -> 19:11:39)
      -> DDoS-RSTFINFlood14_capped_300k.pcap  

In [5]:
# ═══ TONG KET + KIEM TRA ═══
import pandas as pd

df = pd.DataFrame(rows)
print('=' * 78)
print('KET QUA')
print('=' * 78)
print(df[['file', 'status', 'n_in', 'n_out', 'duration_out_s', 'out_MB', 'tool', 'sec']]
      .to_string(index=False))

tot_in  = sum(os.path.getsize(p['src']) for p in plan) / 1e6
tot_out = df.out_MB.sum()
print(f'\nTong dung luong: {tot_in:,.0f} MB -> {tot_out:,.0f} MB '
      f'(con {100*tot_out/tot_in:.0f}%)')
n_cut = int((df.status == 'DA CAT').sum())
print(f'Da cat {n_cut} file, giu nguyen {int((df.status=="GIU NGUYEN").sum())} file.')

df.to_csv(os.path.join(OUT_DIR, 'capping_summary.csv'), index=False)
print(f'Da luu: {OUT_DIR}/capping_summary.csv')

# ── KIEM TRA: file cat ra co doc duoc khong + goi lien tuc khong ──
print('\n' + '=' * 78)
print('KIEM TRA FILE DAU RA')
print('=' * 78)
bad = []
for _, r in df.iterrows():
    if r.status in ('ERROR', 'BO QUA'):
        continue
    p = os.path.join(OUT_DIR, r.out_file)
    try:
        info = pcap_info(p, max_scan=1000)
        ok = info.get('n_packets', 0) > 0
    except Exception as e:
        ok = False
    if not ok:
        bad.append(r.out_file)
print(f'  Doc lai duoc bang dpkt : {"TAT CA OK" if not bad else f"LOI: {bad}"}')

# Do phu thoi gian — canh bao truong hop cat mat phan chinh cua tan cong cham
print('\n  Do phu thoi gian cua phan giu lai:')
for _, r in df.iterrows():
    if r.status != 'DA CAT':
        continue
    d = r.duration_out_s
    warn = '   <-- RAT NGAN, kiem tra lai file nay' if d < 30 else ''
    print(f'    {r.file[:38]:<40}{d:>9.1f}s{warn}')
print('\n  (Neu mot tan cong CHAM nhu SlowLoris chi phu vai giay, phan giu lai')
print('   co the la giai doan thiet lap chu chua phai tan cong chinh.')
print('   Khi do can nang MAX_PACKETS rieng cho file do.)')

print('\n' + '=' * 78)
print('XONG. Tai cac file *_capped_300k.pcap tu tab Output,')
print('tao Dataset moi, roi chay lai Viec 3 (extract & inference) tren tap nay.')
print('LUU Y: predictions_* cu KHONG dung chung duoc — phai chay lai Viec 3.')
print('=' * 78)

KET QUA
                          file     status     n_in  n_out  duration_out_s  out_MB tool  sec
         BrowserHijacking.pcap GIU NGUYEN    59821  59821         1559.99   35.04 copy  0.7
         DDoS-HTTP_Flood-.pcap     DA CAT >300,000 300000          292.29  117.49 dpkt  3.6
        DDoS-ICMP_Flood23.pcap     DA CAT >300,000 300000           11.96   22.88 dpkt  2.7
DDoS-ICMP_Fragmentation16.pcap     DA CAT >300,000 300000       580244.63  265.83 dpkt  3.9
      DDoS-PSHACK_Flood12.pcap     DA CAT >300,000 300000           13.66   22.83 dpkt  2.6
       DDoS-RSTFINFlood14.pcap     DA CAT >300,000 300000           14.72   22.82 dpkt  2.7
DDoS-SynonymousIP_Flood10.pcap     DA CAT >300,000 300000           14.71   22.84 dpkt  2.7
 DDoS-UDP_Fragmentation10.pcap     DA CAT >300,000 300000          565.62  276.69 dpkt  3.9
           DoS-HTTP_Flood.pcap     DA CAT >300,000 300000          219.70  258.70 dpkt  4.0
  Mirai-greip_flood15 (1).pcap     DA CAT >300,000 300000          145.1